# Imports mais criação de DBs

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StringType, IntegerType, DecimalType, DateType
)

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS gold;

# 1º Projeto - Área de Logística (Vendas por Localidade)


## 1.1 Tabela gold.ft_vendas_consumidor_local

Primeiro são lidas silver.ft_pedido_total e silver.ft_consumidores e feito o join pelo id_consumidor, juntando o valor total do pedido com cidade e estado do cliente.

Na seleção das colunas já tem o ajuste de tipos.

id_pedido e id_consumidor como STRING;

valor_total_pedido_brl vindo de valor_total_pago_brl como DECIMAL(12,2);

cidade e estado como STRING;

data_pedido como DATE.

A tabela é pensada para ter uma linha por pedido, sem consolidar nada aqui, justamente para manter o histórico de pedidos. O consolidado por localidade vai ficar só na view depois. No fim, a tabela é salva como gold.ft_vendas_consumidor_local e mostrado um display com limit(10) para conferir.

In [0]:
ft_pedido_total = spark.table("silver.ft_pedido_total")
ft_consumidores = spark.table("silver.ft_consumidores")

ft_vendas_consumidor_local = (
    ft_pedido_total.alias("p")
    .join(
        ft_consumidores.alias("c"),
        on=F.col("p.id_consumidor") == F.col("c.id_consumidor"),
        how="inner"
    )
    .select(
        F.col("p.id_pedido").cast(StringType()).alias("id_pedido"),
        F.col("p.id_consumidor").cast(StringType()).alias("id_consumidor"),
        F.col("p.valor_total_pago_brl")
            .cast(DecimalType(12, 2))
            .alias("valor_total_pedido_brl"),
        F.col("c.cidade").cast(StringType()).alias("cidade"),
        F.col("c.estado").cast(StringType()).alias("estado"),
        F.col("p.data_pedido").cast(DateType()).alias("data_pedido"),
    )
)

ft_vendas_consumidor_local.write.mode("overwrite").saveAsTable("gold.ft_vendas_consumidor_local")
display(ft_vendas_consumidor_local.limit(10))

## 1.2 View gold.view_total_compras_por_consumidor + query

A consulta agrupa por cidade e estado e calcula:

quantidade_vendas usando COUNT(*), que representa quantos pedidos foram feitos em cada localidade;

valor_total_localidade usando SUM(valor_total_pedido_brl), somando o valor total dos pedidos em BRL.

Agrupando os dados por estado e cidade.

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW gold.view_total_compras_por_consumidor AS
SELECT
    cidade,
    estado,
    COUNT(*) AS quantidade_vendas,
    SUM(valor_total_pedido_brl) AS valor_total_localidade
FROM gold.ft_vendas_consumidor_local
GROUP BY cidade, estado
ORDER BY valor_total_localidade DESC
""")

In [0]:
df_view = spark.table("gold.view_total_compras_por_consumidor")
display(df_view.limit(10))


Aqui é feita uma consulta em cima da view gold.view_total_compras_por_consumidor para responder à pergunta de negócio sobre total de vendas por estado.

A query agrupa por estado, soma a coluna quantidade_vendas para calcular o total de vendas em cada um (total_de_vendas_por_estado) e ordena o resultado do maior para o menor, facilitando enxergar quais estados têm maior volume de pedidos.


In [0]:
%sql
SELECT
    estado,
    SUM(quantidade_vendas) AS total_de_vendas_por_estado
FROM gold.view_total_compras_por_consumidor
GROUP BY estado
ORDER BY total_de_vendas_por_estado DESC;

# 2º Projeto - Área de Logística (Análise de Atrasos de Entregas)


## 2.1 Tabela gold.ft_atrasos_pedidos_local_vendedor

São lidas as tabelas silver.ft_pedidos, silver.ft_consumidores e silver.ft_itens_pedidos.

Depois, é feito o join entre pedidos e consumidores por id_consumidor e depois com itens por id_pedido, para juntar informações do pedido, do cliente e do vendedor em uma única tabela.

Na seleção das colunas, os campos são ajustados para o formato pedido no enunciado:
id_pedido, id_vendedor e id_consumidor como string, os tempos de entrega (tempo_entrega_dias e tempo_entrega_estimado_dias) como inteiro, além de entrega_no_prazo, cidade e estado.

A ordenação por tempo_entrega_dias em ordem decrescente ajuda a visualizar primeiro os pedidos com maior tempo de entrega.

Por fim, o resultado é gravado como tabela física na camada gold com o nome gold.

In [0]:
df_pedidos = spark.table("silver.ft_pedidos")
df_consumidores = spark.table("silver.ft_consumidores")
df_itens = spark.table("silver.ft_itens_pedidos")

df_atrasos = (
    df_pedidos.alias("p")
    .join(df_consumidores.alias("c"), "id_consumidor", "left")
    .join(df_itens.alias("i"), "id_pedido", "left")
    .select(
        F.col("p.id_pedido").cast("string").alias("id_pedido"),
        F.col("i.id_vendedor").cast("string").alias("id_vendedor"),
        F.col("p.id_consumidor").cast("string").alias("id_consumidor"),
        F.col("p.entrega_no_prazo").alias("entrega_no_prazo"),
        F.col("p.tempo_entrega_dias").cast("int").alias("tempo_entrega_dias"),
        F.col("p.tempo_entrega_estimado_dias")
            .cast("int")
            .alias("tempo_entrega_estimado_dias"),
        F.col("c.cidade").alias("cidade"),
        F.col("c.estado").alias("estado")
    )
    .orderBy(F.col("tempo_entrega_dias").desc())
)

(df_atrasos
    .write
    .mode("overwrite")
    .saveAsTable("gold.ft_atrasos_pedidos_local_vendedor")
)

display(df_atrasos.limit(10))

## 2.2 Criação das Views Analíticas

### 2.2.1 gold.view_tempo_medio_entrega_localidade

Nesta célula é criada a view gold.view_tempo_medio_entrega_localidade a partir da tabela gold.ft_atrasos_pedidos_local_vendedor.

A consulta agrupa por cidade e estado e calcula a média do tempo real de entrega (tempo_medio_entrega) e a média do tempo estimado (tempo_medio_estimado), ambas arredondadas com 2 casas decimais.

Em seguida, a coluna entrega_maior_que_estimado indica, para cada localidade, se o tempo médio real de entrega ficou maior que o tempo médio estimado (SIM ou NAO).

No final, os resultados são ordenados pelo maior tempo médio de entrega, o que ajuda a identificar rapidamente as regiões com piores tempos de entrega.

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW gold.view_tempo_medio_entrega_localidade AS
SELECT
    cidade,
    estado,
    ROUND(AVG(tempo_entrega_dias), 2) AS tempo_medio_entrega,
    ROUND(AVG(tempo_entrega_estimado_dias), 2) AS tempo_medio_estimado,
    CASE
        WHEN ROUND(AVG(tempo_entrega_dias), 2) > ROUND(AVG(tempo_entrega_estimado_dias), 2)
            THEN 'SIM'
        ELSE 'NAO'
    END AS entrega_maior_que_estimado
FROM gold.ft_atrasos_pedidos_local_vendedor
GROUP BY cidade, estado
ORDER BY tempo_medio_entrega DESC
""")

In [0]:
df_view = spark.table("gold.view_tempo_medio_entrega_localidade")
display(df_view.limit(10))

### 2.2.2 gold.view_vendedor_pontualidade

Aqui é criada a view gold.view_vendedor_pontualidade, focada em avaliar a pontualidade de cada vendedor.

Primeiro, o CTE base seleciona apenas id_vendedor e entrega_no_prazo da tabela gold.

ft_atrasos_pedidos_local_vendedor, filtrando os casos em que id_vendedor não é nulo.

Em seguida, a query principal agrupa por vendedor e calcula: o total de pedidos (total_pedidos), o total de pedidos atrasados (total_atrasados, considerando entrega_no_prazo = 'Não') e o percentual_atraso, que é a proporção de pedidos atrasados em relação ao total, em percentual e com 2 casas decimais.

Por fim, os vendedores são ordenados pelo número de atrasos em ordem decrescente, facilitando a identificação de quem está mais associado a problemas de pontualidade.

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW gold.view_vendedor_pontualidade AS
WITH base AS (
    SELECT
        id_vendedor,
        entrega_no_prazo
    FROM gold.ft_atrasos_pedidos_local_vendedor
    WHERE id_vendedor IS NOT NULL
)
SELECT
    id_vendedor,
    COUNT(*) AS total_pedidos,
    SUM(CASE WHEN entrega_no_prazo = 'Não' THEN 1 ELSE 0 END) AS total_atrasados,
    ROUND(
        100.0 * SUM(CASE WHEN entrega_no_prazo = 'Não' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS percentual_atraso
FROM base
GROUP BY id_vendedor
ORDER BY total_atrasados DESC
""")


In [0]:
df_view = spark.table("gold.view_vendedor_pontualidade")
display(df_view.limit(10))

# 3º Projeto — Área Comercial

## 3.1 Criação da Dimensão de Tempo - gold.dm_tempo

Primeiro, são lidos os pedidos da tabela silver.ft_pedidos e extraídas as datas de compra a partir da coluna pedido_compra_timestamp.

A partir disso, é calculada a menor e a maior data de pedido. Com essas duas datas, é gerada uma sequência diária usando sequence e explode, criando uma linha para cada dia entre o início e o fim do período. Essa coluna vira a sk_tempo.

Em seguida, são criadas as colunas derivadas dessa data: ano, trimestre, mês, semana do ano, dia do mês, número do dia da semana (dia_da_semana_num) e os nomes do dia e do mês em português usando date_format. A coluna eh_fim_de_semana marca com “Sim” os dias em que o número do dia da semana é 1 ou 7 (domingo ou sábado) e “Não” nos demais casos.

Por fim, a dimensão é salva como tabela gold.dm_tempo na camada gold e é feito um display com limit(10) para visualizar algumas linhas de exemplo.

In [0]:
df_pedidos = spark.table("silver.ft_pedidos")

datas = df_pedidos.select(
    F.to_date("pedido_compra_timestamp").alias("data_pedido")
)

min_data, max_data = datas.agg(
    F.min("data_pedido"),
    F.max("data_pedido")
).first()

# Cria sequência de datas
df_tempo = (
    spark
    .createDataFrame([(1,)], ["dummy"])
    .select(
        F.explode(
            F.sequence(
                F.lit(min_data),
                F.lit(max_data),
                F.expr("interval 1 day")
            )
        ).alias("sk_tempo")
    )
)

df_tempo = (
    df_tempo
    .withColumn("ano", F.year("sk_tempo"))
    .withColumn("trimestre", F.quarter("sk_tempo"))
    .withColumn("mes", F.month("sk_tempo"))
    .withColumn("semana_do_ano", F.weekofyear("sk_tempo"))
    .withColumn("dia", F.dayofmonth("sk_tempo"))
    .withColumn("dia_da_semana_num", F.dayofweek("sk_tempo"))  # 1=Dom, 7=Sab
    .withColumn(
        "dia_da_semana_nome",
        F.date_format("sk_tempo", "EEEE") 
    )
    .withColumn(
        "mes_nome",
        F.date_format("sk_tempo", "MMMM")
    )
    .withColumn(
        "eh_fim_de_semana",
        F.when(F.col("dia_da_semana_num").isin(1, 7), "Sim").otherwise("Não")
    )
)

(df_tempo
    .write
    .mode("overwrite")
    .saveAsTable("gold.dm_tempo")
)

display(df_tempo.limit(10))

## 3.2 Fato gold.ft_vendas_geral

Primeiro são lidas as tabelas silver.ft_itens_pedidos, silver.ft_pedidos, silver.dm_cotacao_dolar e silver.ft_avaliacoes_pedidos.

Em seguida, é calculada a média de avaliação por pedido em df_avaliacao_por_pedido, agrupando por id_pedido. Depois, tudo é combinado em df_vendas_geral: os itens são conectados aos pedidos, a cotação do dólar é associada pela data do pedido (pedido_compra_timestamp) e as avaliações são ligadas por id_pedido.

Na seleção das colunas, são montados os campos pedidos no enunciado: identificadores (id_pedido, id_item, fk_cliente, fk_produto, fk_vendedor), a data da compra como fk_tempo, status, tempo de entrega e se foi entregue no prazo. Também são calculados os valores do produto, frete e total em BRL, bem como seus equivalentes em USD usando a cotacao_dolar.

Por fim, entra a média de avaliação do pedido (avaliacao_pedido). O resultado é gravado como tabela gold.ft_vendas_geral na camada gold e exibido um sample para conferência.

In [0]:
df_itens = spark.table("silver.ft_itens_pedidos")
df_pedidos = spark.table("silver.ft_pedidos")
df_cotacao = spark.table("silver.dm_cotacao_dolar")
df_avaliacoes = spark.table("silver.ft_avaliacoes_pedidos")

# média de avaliação por pedido
df_avaliacao_por_pedido = (
    df_avaliacoes
    .groupBy("id_pedido")
    .agg(F.avg("avaliacao").alias("avaliacao_pedido"))
)

# Agora juntando tudo pra ter o resultado final
df_vendas_geral = (
    df_itens.alias("i")
    .join(df_pedidos.alias("p"), "id_pedido")
    .join(
        df_cotacao.alias("d"),
        F.to_date("p.pedido_compra_timestamp") == F.col("d.data"),
        "left"
    )
    .join(df_avaliacao_por_pedido.alias("r"), "id_pedido", "left")
    .select(
        F.col("i.id_pedido").cast("string").alias("id_pedido"),
        F.col("i.id_item").cast("string").alias("id_item"),
        F.col("p.id_consumidor").cast("string").alias("fk_cliente"),
        F.col("i.id_produto").cast("string").alias("fk_produto"),
        F.col("i.id_vendedor").cast("string").alias("fk_vendedor"),
        F.to_date("p.pedido_compra_timestamp").alias("fk_tempo"),
        F.col("p.status").alias("status_pedido"),
        F.col("p.tempo_entrega_dias").cast("int").alias("tempo_entrega_dias"),
        F.col("p.entrega_no_prazo").alias("entrega_no_prazo"),
        F.col("i.preco_brl").cast("decimal(12,2)").alias("valor_produto_brl"),
        F.col("i.preco_frete").cast("decimal(12,2)").alias("valor_frete_brl"),
        (F.col("i.preco_brl") + F.col("i.preco_frete"))
            .cast("decimal(12,2)")
            .alias("valor_total_item_brl"),
        (F.col("i.preco_brl") / F.col("d.cotacao_dolar"))
            .alias("valor_produto_usd"),
        (F.col("i.preco_frete") / F.col("d.cotacao_dolar"))
            .alias("valor_frete_usd"),
        ((F.col("i.preco_brl") + F.col("i.preco_frete")) / F.col("d.cotacao_dolar"))
            .alias("valor_total_item_usd"),
        F.col("d.cotacao_dolar").alias("cotacao_dolar"),
        F.col("r.avaliacao_pedido").cast("decimal(3,2)").alias("avaliacao_pedido")
    )
)

(df_vendas_geral
    .write
    .mode("overwrite")
    .saveAsTable("gold.ft_vendas_geral")
)

display(df_tempo.limit(10))

## 3.3 Criação da view gold.view_vendas_por_periodo

A consulta junta a fato gold.ft_vendas_geral com a dimensão de tempo gold.dm_tempo usando fk_tempo = sk_tempo e agrega por ano, trimestre, mês, nome do mês, dia e número do dia da semana.

São calculados: total de pedidos distintos (total_pedidos), total de itens vendidos (total_itens), receita total em BRL e USD (receita_total_brl e receita_total_usd), além do ticket médio por item em reais (ticket_medio_brl) e a avaliação média dos pedidos (avaliacao_media). Essa view serve como base para análises temporais e dashboards de desempenho de vendas.

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW gold.view_vendas_por_periodo AS
SELECT
    t.ano,
    t.trimestre,
    t.mes,
    t.mes_nome,
    t.dia,
    t.dia_da_semana_num,
    COUNT(DISTINCT f.id_pedido) AS total_pedidos,
    COUNT(*) AS total_itens,
    SUM(f.valor_total_item_brl) AS receita_total_brl,
    ROUND(SUM(f.valor_total_item_usd),2) AS receita_total_usd,
    -- ticket médio por item em BRL
    ROUND(
        SUM(f.valor_total_item_brl) / NULLIF(COUNT(*), 0),
        2
    ) AS ticket_medio_brl,
    ROUND(AVG(f.avaliacao_pedido),2) AS avaliacao_media
FROM gold.ft_vendas_geral f
JOIN gold.dm_tempo t
  ON f.fk_tempo = t.sk_tempo
GROUP BY
    t.ano, t.trimestre, t.mes, t.mes_nome,
    t.dia, t.dia_da_semana_num
""")

In [0]:
df_view = spark.table("gold.view_vendas_por_periodo")
display(df_view.limit(10))

### 3.3.1 Consultas analíticas

Esta consulta usa a view gold.view_vendas_por_periodo para descobrir qual dia da semana gera maior receita em reais.

Os dados são agrupados por dia_da_semana_num e somados em receita_total_brl.

Em seguida, os resultados são ordenados em ordem decrescente de receita e é retornado apenas o primeiro registro (LIMIT 1), que representa o dia da semana com maior faturamento total.

In [0]:
maior_receita_dia = spark.sql("""
SELECT
    dia_da_semana_num,
    SUM(receita_total_brl) AS receita_total_brl
FROM gold.view_vendas_por_periodo
GROUP BY dia_da_semana_num
ORDER BY receita_total_brl DESC
LIMIT 1;
""")

display(maior_receita_dia)


Primeiro, o CTE ultimo_ano pega o maior ano presente na view. Depois, a consulta principal filtra esse ano, agrupa por ano, mês e nome do mês e calcula a média do ticket_medio_brl.

Por fim, os meses são ordenados pelo ticket médio em ordem decrescente e é retornado apenas o primeiro, que representa o mês com maior ticket médio no último ano.

In [0]:
maior_ticket_medio = spark.sql("""
WITH ultimo_ano AS (
    SELECT MAX(ano) AS ano_max
    FROM gold.view_vendas_por_periodo
)
SELECT
    v.ano,
    v.mes,
    v.mes_nome,
    ROUND(AVG(v.ticket_medio_brl), 2) AS ticket_medio_brl
FROM gold.view_vendas_por_periodo v
CROSS JOIN ultimo_ano u
WHERE v.ano = u.ano_max
GROUP BY v.ano, v.mes, v.mes_nome
ORDER BY ticket_medio_brl DESC
LIMIT 1
""")

display(maior_ticket_medio)


## 3.4 View gold.view_top_produto

A consulta usa a fato gold.ft_vendas_geral juntando com silver.ft_produtos pela chave do produto (fk_produto = id_produto).

Depois agrupa por id_produto e categoria_produto e calcula: quantidade total vendida (quantidade_vendida), número de pedidos distintos (total_pedidos), receita total em BRL e USD (receita_brl e receita_usd), preço médio unitário em BRL (preco_medio_brl), avaliação média dos pedidos (avaliacao_media) e o peso médio do produto em gramas (peso_medio_gramas).

No final, os produtos são ordenados pela receita em BRL em ordem decrescente, destacando os itens que mais geram faturamento.

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW gold.view_top_produto AS
SELECT
    p.id_produto,
    p.categoria_produto,
    COUNT(*) AS quantidade_vendida,
    COUNT(DISTINCT f.id_pedido) AS total_pedidos,
    ROUND(SUM(f.valor_total_item_brl),2) AS receita_brl,
    ROUND(SUM(f.valor_total_item_usd),2) AS receita_usd,
    ROUND(AVG(f.valor_produto_brl),2) AS preco_medio_brl,
    ROUND(AVG(f.avaliacao_pedido),2) AS avaliacao_media,
    AVG(p.peso_produto_gramas) AS peso_medio_gramas
FROM gold.ft_vendas_geral f
JOIN silver.ft_produtos p
  ON f.fk_produto = p.id_produto
GROUP BY
    p.id_produto,
    p.categoria_produto
ORDER BY receita_brl DESC
""")

In [0]:
df_view = spark.table("gold.view_top_produto")
display(df_view.limit(10))

## 3.5 View gold.view_vendas_produtos_esteticos (com CTE)

No CTE base, são combinadas a fato gold.ft_vendas_geral, a dimensão de produtos silver.ft_produtos e a dimensão de tempo gold.dm_tempo.

A junção usa fk_produto = id_produto e fk_tempo = sk_tempo, filtrando apenas produtos cuja categoria começa com "fashion".
Na seleção final, os dados são agregados por ano, mês e categoria de produto.

São calculados: o total de pedidos distintos (total_pedidos), o total de itens vendidos (total_itens_vendidos), a receita total em BRL e USD (receita_total_brl e receita_total_usd), o ticket médio por item em BRL e USD (ticket_medio_brl e ticket_medio_usd) e a avaliação média dos pedidos (avaliacao_media).

O resultado é ordenado por ano, mês e categoria, permitindo acompanhar a evolução mensal de volume, receita e satisfação dos clientes nessa categoria específica.

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW gold.view_vendas_produtos_esteticos AS
WITH base AS (
    SELECT
        t.ano,
        t.mes,
        p.categoria_produto,
        f.id_pedido,
        f.id_item,
        f.valor_total_item_brl,
        f.valor_total_item_usd,
        f.avaliacao_pedido
    FROM gold.ft_vendas_geral f
    JOIN silver.ft_produtos p
      ON f.fk_produto = p.id_produto
    JOIN gold.dm_tempo t
      ON f.fk_tempo = t.sk_tempo
    WHERE p.categoria_produto LIKE 'fashion%'
)
SELECT
    ano,
    mes,
    categoria_produto,
    COUNT(DISTINCT id_pedido) AS total_pedidos,
    COUNT(*) AS total_itens_vendidos,
    ROUND(SUM(valor_total_item_brl),2) AS receita_total_brl,
    ROUND(SUM(valor_total_item_usd),2) AS receita_total_usd,
    ROUND(
        SUM(valor_total_item_brl) / NULLIF(COUNT(*), 0),
        2
    ) AS ticket_medio_brl,
    ROUND(
        SUM(valor_total_item_usd) / NULLIF(COUNT(*), 0),
        2
    ) AS ticket_medio_usd,
    ROUND(AVG(avaliacao_pedido),2) AS avaliacao_media
FROM base
GROUP BY
    ano,
    mes,
    categoria_produto
ORDER BY ano, mes, categoria_produto ASC
""")

In [0]:
df_view = spark.table("gold.view_vendas_produtos_esteticos")
display(df_view.limit(10))